# 古い対話を要約して長い対話を続ける

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/21_dialogue/chat_summary_memory.ipynb)

NLP クックブック 21-06 ／ 対話システム

長い対話で、古いターンを単に捨てると大事な情報を忘れてしまう。捨てる前に要約して、記憶として持ち越したい。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

def generate(messages, max_new_tokens=100):
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

class SummaryChat:
    def __init__(self, system, keep_turns=2):
        self.system, self.summary, self.history, self.keep = system, "", [], keep_turns * 2

    def compress(self):
        """直近 keep_turns ターンより古い部分を、これまでの要約と合わせて要約し直す"""
        old, self.history = self.history[:-self.keep], self.history[-self.keep:]
        dialogue = "\n".join(f"{'ユーザー' if m['role'] == 'user' else 'アシスタント'}: {m['content']}" for m in old)
        prompt = ("次の「これまでの要約」と「対話」から、ユーザーの名前・予定・好みと、既に紹介した場所を漏らさずに、3 文以内の新しい要約を書いてください。"
                  "対話に出てこないことは書かないでください。\n\n"
                  f"# これまでの要約\n{self.summary or '(なし)'}\n\n# 対話\n{dialogue}")
        self.summary = generate([{"role": "user", "content": prompt}], max_new_tokens=120)

    def say(self, text):
        self.history.append({"role": "user", "content": text})
        if len(self.history) > self.keep + 2:
            self.compress()
        system = self.system + (f"\n\n# これまでの対話で分かっていること\n{self.summary}" if self.summary else "")
        reply = generate([{"role": "system", "content": system}] + self.history, max_new_tokens=60)
        self.history.append({"role": "assistant", "content": reply})
        return reply

chat = SummaryChat("あなたは旅行の相談に乗るアシスタントです。1〜2 文で短く答えてください。"
                   "「これまでの対話で分かっていること」は、あなたがユーザーから既に聞いた内容です。覚えているものとして答えてください。")
for user in ["私の名前は田中です。来月、京都に 2 泊で行く予定です。", "混雑が苦手なので、静かな場所が好きです。", "秋におすすめのお寺を 1 つ教えてください。",
             "そこへは京都駅からどう行けばいいですか？", "近くでお昼を食べるなら何がいいですか？", "ところで、私の名前と、どんな場所が好きかを覚えていますか？"]:
    print(f"あなた: {user}\nボット: {chat.say(user)}")
    if chat.summary:
        print(f"   [要約] {chat.summary}")

- 直近の数ターンはそのまま残し、それより古い部分だけを要約に置き換える。最近のやりとりは細部まで、昔のことは要点だけ覚えている状態になる。
- 要約は LLM の呼び出しが 1 回増える。毎ターンではなく、履歴が一定の長さを超えたときだけ行う。
- 要約を重ねるうちに細部は落ちていく。確実に覚えておきたい事実 (名前、日程、決定事項) は、要約とは別に項目として抽出して保存する方法もある。
- 過去の対話を検索できる形で保存しておき、関係する部分だけを取り出して入力に加える方法 (RAG と同じ考え方) もよく使われる。